03 - Baseline Model

Goal of this notebook:

- load the processed train / validation / test datasets
- build a small leakage-safe feature set
- train a Logistic Regression baseline
- evaluate with metrics that matter for imbalanced fraud data
- choose a decision threshold using validation data

This notebook is intentionally simple and learning-first.


# Importing the libraries and Setup Config


In [19]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime

import joblib
import sklearn


In [20]:
# ColumnTransformer selects specific columns, applies a transformer to each group, combines the results into one feature matrix
from sklearn.compose import ColumnTransformer  # why this?
from sklearn.impute import (
    SimpleImputer,
)  # for handling missing values, mean imputation and median imputation
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import (
    Pipeline,
)  # to chain multiple data-processing and modeling steps into a single object

# One‑hot encoding is a technique used in data science to convert categorical variables into numerical form so machine‑learning models can use them
# StandardScaler is a feature‑scaling tool in scikit‑learn that standardizes numerical data so each feature has Mean = 0 and Standard deviation = 1
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [21]:
import warnings

from sklearn.exceptions import ConvergenceWarning

In [22]:
PROJECT_DIR = Path("..").resolve()
DATA_DIR = PROJECT_DIR / "Data"
PROCESSED_DIR = DATA_DIR / "processed"

TRAIN_PATH = PROCESSED_DIR / "train.parquet"
TEST_PATH = PROCESSED_DIR / "test.parquet"

print(f"PROJECT DIRECTORY : {PROJECT_DIR} | EXISTS : {PROJECT_DIR.exists()}")
print(f"TRAIN PATH : {PROJECT_DIR} | EXISTS : {TRAIN_PATH.exists()}")
print(f"TEST PATH : {PROJECT_DIR} | EXISTS : {TEST_PATH.exists()}")

PROJECT DIRECTORY : C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection | EXISTS : True
TRAIN PATH : C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection | EXISTS : True
TEST PATH : C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection | EXISTS : True


In [23]:
MODEL_DIR = PROJECT_DIR / "artifacts/models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "sgd_baseline_v1.joblib"


# Data setup


In [24]:
train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("TRAIN SHAPE :", train_df.shape)
print("TEST SHAPE :", test_df.shape)

TRAIN SHAPE : (3731672, 13)
TEST SHAPE : (1346673, 13)


In [25]:
train_df.columns

Index(['transaction_id', 'transaction_timestamp', 'from_bank_id',
       'from_account_id', 'to_bank_id', 'to_account_id', 'amount_received',
       'receiving_currency', 'amount_paid', 'payment_currency',
       'payment_format', 'is_laundering', 'transaction_date'],
      dtype='str')

# Helper Functions


## EDA


In [26]:
def target_summary(name, df):
    count = (
        df["is_laundering"].value_counts(dropna=False).sort_index().to_frame("count")
    )
    count["pct"] = (count["count"] / len(df) * 100).round(4)
    print("-" * 40)
    print(name)
    print("Summary: \n", count)
    print("-" * 40)

## Feature Engineering


In [27]:
def add_basic_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # timestamp is datetime
    df["transaction_timestamp"] = pd.to_datetime(
        df["transaction_timestamp"], errors="coerce"
    )

    # hour, day and is_weekend from timestamp
    df["hour_of_day"] = df["transaction_timestamp"].dt.hour
    df["day_of_week"] = df["transaction_timestamp"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

    # converting amount_received and ammount_paid into numeric
    df["amount_received"] = pd.to_numeric(df["amount_received"], errors="coerce")
    df["amount_paid"] = pd.to_numeric(df["amount_paid"], errors="coerce")

    # same_currency flag if receiving and sending payment currency is same
    df["same_currency_flag"] = (
        df["receiving_currency"].astype(str) == df["payment_currency"].astype(str)
    ).astype(int)

    # same_bank flag if receiving and sending payment currbank is same
    df["same_bank_flag"] = (
        df["from_bank_id"].astype(str) == df["to_bank_id"].astype(str)
    ).astype(int)

    # avoid log(0) - because log(0) is undefined and using np.log1p makes log(0) -> log(1) because [log(1+x)]. clip(lower = 0) makes any value below 0 is set to 0
    df["log_amount_received"] = np.log1p(df["amount_received"].clip(lower=0))
    df["log_amount_paid"] = np.log1p(df["amount_paid"].clip(lower=0))

    return df

## core evaluation into one function

In [28]:
def evaluate_baseline(
    model,
    X,
    y,
    threshold=0.50,
):
    """Evaluate ranking quality and threshold-based fraud performance."""

    probabilities = model.predict_proba(X)[:, 1]

    predictions = (probabilities >= threshold).astype(int)

    pr_auc = average_precision_score(y, probabilities)

    roc_auc = roc_auc_score(y, probabilities)

    precision = precision_score(y, predictions, zero_division=0)

    recall = recall_score(y, predictions, zero_division=0)

    f1 = f1_score(y, predictions, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y, predictions).ravel()

    positive_rate = y.mean()
    naive_pr_auc = positive_rate

    total_alerts = tp + fp
    total_transactions = len(y)

    alert_rate = total_alerts / total_transactions

    false_positive_rate = fp / (fp + tn)

    print("=" * 60)
    print(f"EVALUATION @ THRESHOLD {threshold:.2f}")
    print("=" * 60)

    print("\nRanking Metrics")
    print("-" * 40)
    print(f"PR-AUC:            {pr_auc:.6f}")
    print(f"ROC-AUC:           {roc_auc:.6f}")
    print(f"Naive PR-AUC:      {naive_pr_auc:.6f}")
    print(f"PR-AUC lift:       {pr_auc / naive_pr_auc:.2f}x")

    print("\nClassification Metrics")
    print("-" * 40)
    print(f"Precision:         {precision:.6f}")
    print(f"Recall:            {recall:.6f}")
    print(f"F1:                {f1:.6f}")

    print("\nConfusion Matrix")
    print("-" * 40)
    print(f"True negatives:    {tn:,}")
    print(f"False positives:   {fp:,}")
    print(f"False negatives:   {fn:,}")
    print(f"True positives:    {tp:,}")

    print("\nOperational Metrics")
    print("-" * 40)
    print(f"Alerts generated:  {total_alerts:,}")
    print(f"Alert rate:        {alert_rate:.2%}")
    print(f"False-positive rate: {false_positive_rate:.2%}")

    if tp > 0:
        print(f"Alerts / TP:        {total_alerts / tp:.1f}")
        print(f"FP / TP:            {fp / tp:.1f}")

    return {
        "threshold": threshold,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "alert_rate": alert_rate,
        "false_positive_rate": false_positive_rate,
    }

# EDA


In [29]:
target_summary("Training Data", train_df)

----------------------------------------
Training Data
Summary: 
                  count      pct
is_laundering                  
0              3728645  99.9189
1                 3027   0.0811
----------------------------------------


In [30]:
target_summary("Test Data", test_df)

----------------------------------------
Test Data
Summary: 
                  count      pct
is_laundering                  
0              1344523  99.8403
1                 2150   0.1597
----------------------------------------


## Feature engineering

Selected baseline features include:

- raw amount columns
- log amounts
- payment format
- hour of day
- day of week
- weekend flag
- same-currency flag
- same-bank vs cross-bank flag

I will keep the baseline small and row-level only.


In [31]:
train_feat = add_basic_features(train_df)
test_feat = add_basic_features(test_df)

print("-" * 40)
print("Train data - features")
print(train_feat.info())
print("-" * 40)
print("Test data - features")
print(test_feat.info())
print("-" * 40)

----------------------------------------
Train data - features
<class 'pandas.DataFrame'>
RangeIndex: 3731672 entries, 0 to 3731671
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   transaction_id         str           
 1   transaction_timestamp  datetime64[us]
 2   from_bank_id           str           
 3   from_account_id        str           
 4   to_bank_id             str           
 5   to_account_id          str           
 6   amount_received        float64       
 7   receiving_currency     str           
 8   amount_paid            float64       
 9   payment_currency       str           
 10  payment_format         str           
 11  is_laundering          int64         
 12  transaction_date       object        
 13  hour_of_day            int32         
 14  day_of_week            int32         
 15  is_weekend             int64         
 16  same_currency_flag     int64         
 17  same_bank_fla

In [32]:
train_feat.columns

Index(['transaction_id', 'transaction_timestamp', 'from_bank_id',
       'from_account_id', 'to_bank_id', 'to_account_id', 'amount_received',
       'receiving_currency', 'amount_paid', 'payment_currency',
       'payment_format', 'is_laundering', 'transaction_date', 'hour_of_day',
       'day_of_week', 'is_weekend', 'same_currency_flag', 'same_bank_flag',
       'log_amount_received', 'log_amount_paid'],
      dtype='str')

In [33]:
TRAIN_FILE = PROCESSED_DIR / "training_data.parquet"

In [34]:
train_feat.to_parquet(TRAIN_FILE, index=False)

In [35]:
TEST_FILE = PROCESSED_DIR / "testing_data.parquet"

In [36]:
test_feat.to_parquet(TEST_FILE, index=False)

## train - val split for baseline


In [46]:
validation_start = pd.Timestamp("2022-09-06").date()

In [47]:
validation_start

datetime.date(2022, 9, 6)

In [48]:
sgd_train_df = train_feat[train_feat["transaction_date"] < validation_start].copy()

sgd_val_df = train_feat[train_feat["transaction_date"] >= validation_start].copy()

In [49]:
sgd_train_df.shape

(2766832, 20)

In [50]:
sgd_val_df.shape

(964840, 20)

In [51]:
len(sgd_train_df) / len(train_feat) * 100

74.14456576033479

In [52]:
len(sgd_val_df) / len(train_feat) * 100

25.85543423966522

In [53]:
def print_split_summary(name, df):
    print(f"\n{name}")
    print("-" * 40)
    print(f"Rows: {len(df):,}")
    mini = df["transaction_date"].min()
    maxi = df["transaction_date"].max()
    suma = df["is_laundering"].sum()
    meani = df["is_laundering"].mean()

    print(f"Date range: {mini} to {maxi}")
    print(f"Fraud cases: {suma:,}")
    print(f"Fraud rate: {meani:.6%}")


print_split_summary("SGD training data", sgd_train_df)
print_split_summary("SGD validation data", sgd_val_df)


SGD training data
----------------------------------------
Rows: 2,766,832
Date range: 2022-09-01 to 2022-09-05
Fraud cases: 1,999
Fraud rate: 0.072249%

SGD validation data
----------------------------------------
Rows: 964,840
Date range: 2022-09-06 to 2022-09-07
Fraud cases: 1,028
Fraud rate: 0.106546%


## Baseline features


In [54]:
feature_cols = [
    "amount_received",
    "amount_paid",
    "receiving_currency",
    "payment_currency",
    "payment_format",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "same_currency_flag",
    "same_bank_flag",
    "log_amount_received",
    "log_amount_paid",
    "transaction_date",
]

target_col = "is_laundering"

In [55]:
numeric_cols = [
    "amount_received",
    "amount_paid",
    "log_amount_received",
    "log_amount_paid",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "same_currency_flag",
    "same_bank_flag",
]

categorical_cols = ["payment_format", "receiving_currency", "payment_currency"]

In [56]:
X_train = sgd_train_df[feature_cols]
Y_train = sgd_train_df[target_col]

X_val = sgd_val_df[feature_cols]
Y_val = sgd_val_df[target_col]

X_test = test_feat[feature_cols]
Y_test = test_feat[target_col]

print(f"X_train shape: {X_train.shape} | Y_train.shape: {Y_train.shape}")
print(f"X_val shape: {X_val.shape} | Y_val.shape: {Y_val.shape}")
print(f"X_test shape: {X_test.shape} | Y_test.shape: {Y_test.shape}")

X_train shape: (2766832, 13) | Y_train.shape: (2766832,)
X_val shape: (964840, 13) | Y_val.shape: (964840,)
X_test shape: (1346673, 13) | Y_test.shape: (1346673,)


## Preprocessing

For this first baseline:

- numeric columns will be imputed and scaled
- categorical columns will be imputed and one-hot encoded

I know this is not needed for this dataset, since there are missing or null values, following this step because it is good practice to do

Because the classes are extremely imbalanced, I will start with `class_weight="balanced"` for the baseline.


In [57]:
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

In [58]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("OneHotEncoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

In [59]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# Baseline Model


In [60]:
sgd_model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=0.0001,
    class_weight="balanced",
    max_iter=100,
    tol=1e-3,
    n_iter_no_change=5,
    random_state=42,
    average=True,
    early_stopping=False,
)

In [61]:
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", sgd_model),
    ]
)

## Train model


In [62]:
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always", ConvergenceWarning)

    baseline_model.fit(X_train, Y_train)

convergence_warnings = [
    warning
    for warning in caught_warnings
    if issubclass(warning.category, ConvergenceWarning)
]

print("Baseline Model Training Completed")

Baseline Model Training Completed


In [63]:
fitted_sgd = baseline_model.named_steps["classifier"]

print(f"Epochs completed: {fitted_sgd.n_iter_}")
print(f"Maximum epochs:   {fitted_sgd.max_iter}")

if convergence_warnings:
    print("The model did not converge. Increase max_iter.")
else:
    print("No convergence warning was raised.")

Epochs completed: 14
Maximum epochs:   100
No convergence warning was raised.


In [125]:
preprocessor = baseline_model.named_steps["preprocessor"]
classifier = baseline_model.named_steps["classifier"]

feature_names = preprocessor.get_feature_names_out()

coefficients = classifier.coef_[0]

coef_df = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coefficients,
    }
)

coef_df["abs_coefficient"] = np.abs(coef_df["coefficient"])

coef_df = coef_df.sort_values("abs_coefficient", ascending=False)


coef_df.head(30)

,feature,coefficient,abs_coefficient
1,num__amount_paid,65.293130,65.293130
0,num__amount_received,40.501122,40.501122
14,cat__payment_format_Reinvestment,-14.467038,14.467038
15,cat__payment_format_Wire,-13.124250,13.124250
13,cat__payment_format_Credit Card,-12.076694,12.076694
12,cat__payment_format_Cheque,-11.394353,11.394353
11,cat__payment_format_Cash,-10.732276,10.732276
10,cat__payment_format_Bitcoin,-6.055018,6.055018
17,cat__receiving_currency_Bitcoin,-6.006141,6.006141
32,cat__payment_currency_Bitcoin,-6.000874,6.000874


In [126]:
coef_df[coef_df["feature"].str.contains("amount|payment_format", case=False)]

,feature,coefficient,abs_coefficient
1,num__amount_paid,65.293130,65.293130
0,num__amount_received,40.501122,40.501122
14,cat__payment_format_Reinvestment,-14.467038,14.467038
15,cat__payment_format_Wire,-13.124250,13.124250
13,cat__payment_format_Credit Card,-12.076694,12.076694
12,cat__payment_format_Cheque,-11.394353,11.394353
11,cat__payment_format_Cash,-10.732276,10.732276
10,cat__payment_format_Bitcoin,-6.055018,6.055018
9,cat__payment_format_ACH,-3.787619,3.787619
3,num__log_amount_paid,0.868754,0.868754


# Baseline Evaluation

## Validation risk Scores

- Extracting the predicted probability for the positive class (Class 1).
- Why use probabilities instead of hard predictions? 
    - In fraud detection, we care about risk ranking rather than just binary outputs. 
    - Saving these raw probabilities allows us to compare model rankings against future improvements
    - also will help analyze the score distribution before choosing an optimal classification threshold.

In [69]:
train_probabilities = baseline_model.predict_proba(X_train)[:, 1]
val_probabilities = baseline_model.predict_proba(X_val)[:, 1]

## Ranking quality evaluation

In [71]:
train_pr_auc = average_precision_score(Y_train, train_probabilities)
val_pr_auc = average_precision_score(Y_val, val_probabilities)

In [ ]:
train_roc_auc = roc_auc_score(Y_train, train_probabilities)
val_roc_auc = roc_auc_score(Y_val, val_probabilities)

In [73]:
print("Ranking Performance")
print("-" * 50)

print(f"Train PR-AUC:      {train_pr_auc:.6f}")
print(f"Validation PR-AUC: {val_pr_auc:.6f}")

print()

print(f"Train ROC-AUC:      {train_roc_auc:.6f}")
print(f"Validation ROC-AUC: {val_roc_auc:.6f}")

Ranking Performance
--------------------------------------------------
Train PR-AUC:      0.008182
Validation PR-AUC: 0.010280

Train ROC-AUC:      0.897565
Validation ROC-AUC: 0.898973


The Key Findings⚠️:
- Model is performing barely better than random guessing since PR-AUC (~0.010)
    - An score around ~0.01 indicates that the dataset likely has about 1% fraud, and model's predictions have almost zero precision.
- ROC-AUC (~0.898) is giving a false sense of security  
    - it is inflated by the overwhelming majority of non-fraudulent (legitimate) transactions.
    - Since ROC-AUC measures the False Positive Rate and we have a massive pool of legitimate transactions (99% of the data), flagging thousands of not fraud data still yields a tiny False Positive Rate, making ROC-AUC artificially high.
- We do not have an overfitting problem since Train and Validation scores are close:
    - PR-AUC: 0.0081 (Train) vs 0.0102 (Validation)
    - ROC-AUC: 0.8975 (Train) vs 0.8989 (Validation)
    - This means model generalizes consistently. The issue isn't overfitting. The model simply hasn't learned meaningful patterns to separate fraud from non-fraud yet.

## Establishing the benchmark

In [138]:
# Rule of Thumb: In PR-AUC, the baseline for a random model is equal to the positive rate itself
actual_positive_rate = Y_val.mean()

In [139]:
naive_pr_auc = actual_positive_rate


In [140]:
improvement_ratio = val_pr_auc / naive_pr_auc

In [78]:
print("\nClass Imbalance Context")
print("-" * 50)

print(f"Validation positive rate: {actual_positive_rate:.6%}")
print(f"Naive PR-AUC:             {naive_pr_auc:.6f}")
print(f"Model PR-AUC:             {val_pr_auc:.6f}")
print(f"PR-AUC lift:              {improvement_ratio:.2f}x")


Class Imbalance Context
--------------------------------------------------
Validation positive rate: 0.106546%
Naive PR-AUC:             0.001065
Model PR-AUC:             0.010280
PR-AUC lift:              9.65x


- Baseline model is ~9.65 times better than random guessing
- While 9.65x shows that the pipeline works and the model is learning patterns, a PR-AUC of 0.01 still means that if the model flags the top risky transactions, around ~98-99% of the alerts will still be false positives.

## Evaluate the default 0.50 operating point

In [133]:
baseline_threshold = 0.50

val_predictions = (val_probabilities >= baseline_threshold).astype(int)

In [134]:
precision = precision_score(Y_val, val_predictions, zero_division=0)

In [135]:
recall = recall_score(Y_val, val_predictions, zero_division=0)

In [136]:
f1 = f1_score(Y_val, val_predictions, zero_division=0)

In [137]:
tn, fp, fn, tp = confusion_matrix(Y_val, val_predictions).ravel()

In [85]:
print("\nBaseline @ Threshold = 0.50")
print("-" * 50)

print(f"Precision: {precision:.6f}")
print(f"Recall:    {recall:.6f}")
print(f"F1:        {f1:.6f}")

print("\nConfusion Matrix")
print(f"TN: {tn:,}")
print(f"FP: {fp:,}")
print(f"FN: {fn:,}")
print(f"TP: {tp:,}")


Baseline @ Threshold = 0.50
--------------------------------------------------
Precision: 0.008112
Recall:    0.865759
F1:        0.016073

Confusion Matrix
TN: 854,982
FP: 108,830
FN: 138
TP: 890


## metrics that actually describe analyst burden

In [86]:
total_transactions = len(Y_val)

In [87]:
actual_frauds = tp + fn
total_alerts = tp + fp

In [89]:
alert_rate = total_alerts / total_transactions
false_positive_rate = fp / (fp + tn)

In [92]:
alerts_per_true_positive = total_alerts / tp if tp > 0 else float("inf")
false_positives_per_true_positive = fp / tp if tp > 0 else float("inf")


In [93]:
miss_rate = fn / actual_frauds if actual_frauds > 0 else 0

In [94]:
print("\nOperational Burden")
print("-" * 50)

print(f"Total transactions:          {total_transactions:,}")
print(f"Actual frauds:               {actual_frauds:,}")
print(f"Total alerts:                {total_alerts:,}")

print(f"\nAlert rate:                   {alert_rate:.2%}")
print(f"False-positive rate:          {false_positive_rate:.2%}")
print(f"Fraud miss rate:              {miss_rate:.2%}")

print(f"Alerts per detected fraud:    {alerts_per_true_positive:.1f}")

print(f"False alerts per true fraud:  {false_positives_per_true_positive:.1f}")


Operational Burden
--------------------------------------------------
Total transactions:          964,840
Actual frauds:               1,028
Total alerts:                109,720

Alert rate:                   11.37%
False-positive rate:          11.29%
Fraud miss rate:              13.42%
Alerts per detected fraud:    123.3
False alerts per true fraud:  122.3


- Alerts per detected fraud (123.3): For every single fraudulent transaction your team successfully catches, they have to investigate 123 total alerts.
- False alerts per true fraud (122.3):Out of those 123 alerts investigated, 122 of them were completely innocent customers, and only 1 was an actual fraudster.

## threshold analysis

In [107]:
baseline_results = evaluate_baseline(
    baseline_model,
    X_val,
    Y_val,
    threshold=0.99,
)

EVALUATION @ THRESHOLD 0.99

Ranking Metrics
----------------------------------------
PR-AUC:            0.010280
ROC-AUC:           0.898973
Naive PR-AUC:      0.001065
PR-AUC lift:       9.65x

Classification Metrics
----------------------------------------
Precision:         0.012756
Recall:            0.553502
F1:                0.024938

Confusion Matrix
----------------------------------------
True negatives:    919,776
False positives:   44,036
False negatives:   459
True positives:    569

Operational Metrics
----------------------------------------
Alerts generated:  44,605
Alert rate:        4.62%
False-positive rate: 4.57%
Alerts / TP:        78.4
FP / TP:            77.4


- SGD model’s probability scores are heavily concentrated near the high end, so moving the threshold from 0.50 → 0.95 barely changes the operating behavior. Only around 0.99 do we start seeing a major change.
- 
    | Threshold | Precision | Recall | F1 | False Positives | Alerts | Alert Rate |
    | --- | --- | --- | --- | --- | --- | --- |
    | 0.50 | 0.59% | 87.35% | 1.16% | 152,375 | 153,273 | 15.89% |
    | 0.60 | 0.83% | 86.48% | 1.63% | 106,849 | 107,738 | 11.17% |
    | 0.80 | 0.84% | 85.41% | 1.66% | 103,832 | 104,710 | 10.85% |
    | 0.95 | 0.89% | 84.44% | 1.76% | 96,795 | 97,663 | 10.12% |
    | 0.99 | 1.28% | 55.35% | 2.49% | 44,036 | 44,605 | 4.62% |

- Increasing the threshold absolutely helps FP
- But 0.60 → 0.95 hardly changes anything
- lot of both legitimate transactions and fraud transactions are packed into approximately the 0.95–0.99+ score region.
- 0.99 is still nowhere near operationally good. ~77 false alerts, still potentially a very large analyst workload. 459 / 1,028 fraud transactions are now missed since recall is low. That's almost 45% of the fraud.

- next model does not merely need higher recall. 
- next candidate should aim for 
    - Higher PR-AUC
    - Better precision at useful recall
    - Lower alert burden


In [108]:
quantiles = [
    0,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.97,
    0.98,
    0.99,
    0.995,
    0.999,
    1.0,
]

print("Validation probability quantiles")
print("-" * 50)

for q in quantiles:
    print(f"{q:>6.3f}: {np.quantile(val_probabilities, q):.6f}")

Validation probability quantiles
--------------------------------------------------
 0.000: 0.000000
 0.250: 0.016890
 0.500: 0.035752
 0.750: 0.080655
 0.900: 0.953733
 0.950: 0.989138
 0.970: 0.993336
 0.980: 0.995203
 0.990: 0.997451
 0.995: 0.999122
 0.999: 1.000000
 1.000: 1.000000


In [ ]:
fraud_scores = val_probabilities[np.asarray(Y_val) == 1]

legitimate_scores = val_probabilities[np.asarray(Y_val) == 0]

In [ ]:
print("\nFraud score quantiles")
print("-" * 50)

for q in quantiles:
    print(f"{q:>6.3f}: {np.quantile(fraud_scores, q):.6f}")

print("\nLegitimate score quantiles")
print("-" * 50)

for q in quantiles:
    print(f"{q:>6.3f}: {np.quantile(legitimate_scores, q):.6f}")


Fraud score quantiles
--------------------------------------------------
 0.000: 0.002316
 0.250: 0.984014
 0.500: 0.991271
 0.750: 0.994972
 0.900: 0.996719
 0.950: 0.997536
 0.970: 0.997970
 0.980: 0.999285
 0.990: 0.999990
 0.995: 1.000000
 0.999: 1.000000
 1.000: 1.000000

Legitimate score quantiles
--------------------------------------------------
 0.000: 0.000000
 0.250: 0.016876
 0.500: 0.035709
 0.750: 0.080421
 0.900: 0.951377
 0.950: 0.988993
 0.970: 0.993256
 0.980: 0.995166
 0.990: 0.997436
 0.995: 0.999114
 0.999: 1.000000
 1.000: 1.000000


In [110]:
val_decision_scores = baseline_model.decision_function(X_val)

In [111]:
fraud_decision_scores = val_decision_scores[np.asarray(Y_val) == 1]

legitimate_decision_scores = val_decision_scores[np.asarray(Y_val) == 0]

In [112]:
quantiles = [
    0,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.97,
    0.98,
    0.99,
    0.995,
    0.999,
    1.0,
]

print("Fraud decision-score quantiles")
print("-" * 50)

for q in quantiles:
    print(f"{q:>6.3f}: {np.quantile(fraud_decision_scores, q):.6f}")


print("\nLegitimate decision-score quantiles")
print("-" * 50)

for q in quantiles:
    print(f"{q:>6.3f}: {np.quantile(legitimate_decision_scores, q):.6f}")

Fraud decision-score quantiles
--------------------------------------------------
 0.000: -6.065579
 0.250: 4.119967
 0.500: 4.732363
 0.750: 5.287630
 0.900: 5.716398
 0.950: 6.003494
 0.970: 6.197444
 0.980: 7.260212
 0.990: 11.596370
 0.995: 20.367388
 0.999: 364.235817
 1.000: 3028.095744

Legitimate decision-score quantiles
--------------------------------------------------
 0.000: -31.456107
 0.250: -4.064837
 0.500: -3.295980
 0.750: -2.436643
 0.900: 2.973807
 0.950: 4.498119
 0.970: 4.992375
 0.980: 5.327312
 0.990: 5.963680
 0.995: 7.028329
 0.999: 48.037719
 1.000: 146951.968531


## investigate the extreme rows

In [113]:
analysis_df = X_val.copy()

In [114]:
analysis_df["actual_label"] = np.asarray(Y_val)
analysis_df["decision_score"] = val_decision_scores
analysis_df["risk_score"] = val_probabilities

In [115]:
analysis_df

,amount_received,amount_paid,receiving_currency,payment_currency,payment_format,hour_of_day,day_of_week,is_weekend,same_currency_flag,same_bank_flag,log_amount_received,log_amount_paid,transaction_date,actual_label,decision_score,risk_score
2766832,114.95,114.95,US Dollar,US Dollar,Credit Card,0,1,0,1,0,4.753159,4.753159,2022-09-06,0,-3.544342,0.028077
2766833,155.95,155.95,Swiss Franc,Swiss Franc,Cheque,0,1,0,1,0,5.055927,5.055927,2022-09-06,0,-3.134950,0.041688
2766834,1358.81,1358.81,US Dollar,US Dollar,Credit Card,0,1,0,1,0,7.215100,7.215100,2022-09-06,0,-2.915553,0.051390
2766835,145.33,145.33,US Dollar,US Dollar,Credit Card,0,1,0,1,0,4.985864,4.985864,2022-09-06,0,-3.484920,0.029744
2766836,16.47,16.47,US Dollar,US Dollar,Credit Card,0,1,0,1,0,2.860485,2.860485,2022-09-06,0,-4.027617,0.017505
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3731667,2804.22,2804.22,Euro,Euro,ACH,23,2,0,1,0,7.939237,7.939237,2022-09-07,0,3.741195,0.976824
3731668,2934716.59,2934716.59,Mexican Peso,Mexican Peso,Wire,23,2,0,1,0,14.892122,14.892122,2022-09-07,0,-4.294136,0.013465
3731669,84.82,84.82,Euro,Euro,Cheque,23,2,0,1,0,4.452252,4.452252,2022-09-07,0,-4.756263,0.008524
3731670,32.21,32.21,US Dollar,US Dollar,Credit Card,23,2,0,1,0,3.502851,3.502851,2022-09-07,0,-4.926472,0.007200


In [121]:
top_false_positives = (
    analysis_df[analysis_df["actual_label"] == 0]
    .sort_values("decision_score", ascending=False)
    .head(20)
)

top_false_positives

,amount_received,amount_paid,receiving_currency,payment_currency,payment_format,hour_of_day,day_of_week,is_weekend,same_currency_flag,same_bank_flag,log_amount_received,log_amount_paid,transaction_date,actual_label,decision_score,risk_score
3129011,1.046302e+12,1.046302e+12,Yen,Yen,ACH,17,1,0,1,0,27.676284,27.676284,2022-09-06,0,146951.968531,1.0
3429861,9.659333e+11,9.659333e+11,Rupee,Rupee,ACH,8,2,0,1,0,27.596361,27.596361,2022-09-07,0,135664.712182,1.0
3241774,1.499632e+11,1.499632e+11,Brazil Real,Brazil Real,ACH,23,1,0,1,0,25.733656,25.733656,2022-09-06,0,21070.151017,1.0
2804711,1.402124e+11,1.402124e+11,Ruble,Ruble,Cheque,1,1,0,1,0,25.666424,25.666424,2022-09-06,0,19692.527456,1.0
3337635,1.402124e+11,1.402124e+11,Ruble,Ruble,Cheque,3,2,0,1,0,25.666424,25.666424,2022-09-07,0,19692.379866,1.0
3503132,7.230977e+10,7.230977e+10,Yen,Yen,ACH,12,2,0,1,0,25.004225,25.004225,2022-09-07,0,10163.575971,1.0
2945051,4.875305e+10,4.875305e+10,Rupee,Rupee,ACH,8,1,0,1,0,24.610034,24.610034,2022-09-06,0,6854.887931,1.0
2804701,1.402124e+11,1.802123e+09,Ruble,US Dollar,ACH,1,1,0,0,1,25.666424,21.312231,2022-09-06,0,6278.044609,1.0
3337815,1.402124e+11,1.802123e+09,Ruble,US Dollar,ACH,3,2,0,0,1,25.666424,21.312231,2022-09-07,0,6277.897019,1.0
3048129,2.537087e+10,2.537087e+10,Rupee,Rupee,ACH,13,1,0,1,0,23.956868,23.956868,2022-09-06,0,3570.702024,1.0


In [117]:
top_true_positives = (
    analysis_df[analysis_df["actual_label"] == 1]
    .sort_values("decision_score", ascending=False)
    .head(20)
)

In [119]:
top_true_positives

,amount_received,amount_paid,receiving_currency,payment_currency,payment_format,hour_of_day,day_of_week,is_weekend,same_currency_flag,same_bank_flag,log_amount_received,log_amount_paid,transaction_date,actual_label,decision_score,risk_score
2809898,2.150069e+10,2.150069e+10,Canadian Dollar,Canadian Dollar,ACH,1,1,0,1,0,23.791351,23.791351,2022-09-06,1,3028.095744,1.000000
3202657,2.597164e+09,2.597164e+09,Saudi Riyal,Saudi Riyal,ACH,21,1,0,1,0,21.677686,21.677686,2022-09-06,1,373.221529,1.000000
3692222,2.449716e+08,2.449716e+08,Rupee,Rupee,ACH,21,2,0,1,0,19.316653,19.316653,2022-09-07,1,40.417377,1.000000
3428097,2.317129e+08,2.317129e+08,Mexican Peso,Mexican Peso,Cash,8,2,0,1,0,19.261010,19.261010,2022-09-07,1,31.996821,1.000000
3041149,1.304713e+08,1.304713e+08,Swiss Franc,Swiss Franc,ACH,13,1,0,1,0,18.686664,18.686664,2022-09-06,1,25.708966,1.000000
3246712,9.445609e+07,9.445609e+07,US Dollar,US Dollar,ACH,23,1,0,1,0,18.363646,18.363646,2022-09-06,1,20.482912,1.000000
3166653,8.970926e+07,8.970926e+07,Swiss Franc,Swiss Franc,ACH,19,1,0,1,0,18.312085,18.312085,2022-09-06,1,19.627180,1.000000
3497217,6.296766e+07,6.296766e+07,Euro,Euro,ACH,11,2,0,1,0,17.958132,17.958132,2022-09-07,1,15.665173,1.000000
3276703,4.615899e+07,4.615899e+07,US Dollar,US Dollar,ACH,0,2,0,1,0,17.647602,17.647602,2022-09-07,1,14.459256,0.999999
3693527,3.895423e+07,3.895423e+07,Saudi Riyal,Saudi Riyal,ACH,22,2,0,1,0,17.477898,17.477898,2022-09-07,1,12.769237,0.999997


In [118]:
lowest_scoring_frauds = (
    analysis_df[analysis_df["actual_label"] == 1]
    .sort_values("decision_score", ascending=True)
    .head(20)
)

lowest_scoring_frauds

,amount_received,amount_paid,receiving_currency,payment_currency,payment_format,hour_of_day,day_of_week,is_weekend,same_currency_flag,same_bank_flag,log_amount_received,log_amount_paid,transaction_date,actual_label,decision_score,risk_score
3347832,0.040213,0.040213,Bitcoin,Bitcoin,Bitcoin,4,2,0,1,1,0.039425,0.039425,2022-09-07,1,-6.065579,0.002316
3226786,14.530000,14.530000,Swiss Franc,Swiss Franc,Credit Card,22,1,0,1,0,2.742774,2.742774,2022-09-06,1,-5.366804,0.004647
3209353,1916.060000,1916.060000,Rupee,Rupee,Credit Card,21,1,0,1,0,7.558548,7.558548,2022-09-06,1,-5.217140,0.005394
3198418,116.170000,116.170000,Euro,Euro,Credit Card,21,1,0,1,0,4.763626,4.763626,2022-09-06,1,-5.211505,0.005424
3559228,56.980000,56.980000,Euro,Euro,Credit Card,15,2,0,1,0,4.060098,4.060098,2022-09-07,1,-5.190057,0.005541
3332163,90.920000,90.920000,UK Pound,UK Pound,Credit Card,3,2,0,1,0,4.520919,4.520919,2022-09-07,1,-4.878365,0.007552
3133609,10.850000,10.850000,Brazil Real,Brazil Real,Credit Card,18,1,0,1,0,2.472328,2.472328,2022-09-06,1,-4.745965,0.008612
3344888,79.060000,79.060000,Euro,Euro,Credit Card,4,2,0,1,0,4.382776,4.382776,2022-09-07,1,-4.628229,0.009677
2924714,108.750000,108.750000,Euro,Euro,Credit Card,7,1,0,1,0,4.698205,4.698205,2022-09-06,1,-4.618021,0.009776
3234844,365.890000,365.890000,Swiss Franc,Swiss Franc,Credit Card,23,1,0,1,0,5.905062,5.905062,2022-09-06,1,-4.602903,0.009923


# Save the Joblib

In [147]:
baseline_artifact = {
    # Complete fitted sklearn Pipeline
    "model": baseline_model,
    # Identity
    "model_name": "SGD Baseline v1",
    "model_type": "SGDClassifier",
    "created_at": datetime.now().isoformat(),
    # Dataset split
    "training_period": {
        "start": "2022-09-01",
        "end": "2022-09-05",
    },
    "validation_period": {
        "start": "2022-09-06",
        "end": "2022-09-07",
    },
    "test_period": {
        "start": "2022-09-08",
        "status": "UNTOUCHED",
    },
    # Input features expected by the pipeline
    "feature_columns": list(X_train.columns),
    # Training information
    "epochs_completed": int(baseline_model.named_steps["classifier"].n_iter_),
    "classifier_params": (baseline_model.named_steps["classifier"].get_params()),
    # Validation ranking metrics
    "validation_metrics": {
        "pr_auc": f"{val_pr_auc:.8f}",
        "roc_auc": f"{val_roc_auc:.8f}",
        "positive_rate": f"{actual_positive_rate:.8f}",
        "naive_pr_auc": f"{naive_pr_auc:.8f}",
        "pr_auc_lift": f"{improvement_ratio:.2f}",
    },
    # 0.50 is explicitly diagnostic, NOT selected
    "diagnostic_threshold": {
        "threshold": 0.50,
        "purpose": "baseline diagnostic only",
        "precision": f"{precision:.6f}",
        "recall": f"{recall:.6f}",
        "f1": f"{f1:.6f}",
        "tn": f"{tn:,}",
        "fp": f"{fp:,}",
        "fn": f"{fn:,}",
        "tp": f"{tp:,}",
    },
    "notes": {
        "threshold_selected": False,
        "probability_calibrated": False,
        "baseline_frozen": True,
        "primary_comparison_metric": "PR-AUC",
    },
    # Environment reproducibility
    "sklearn_version": sklearn.__version__,
}


In [149]:
joblib.dump(
    baseline_artifact,
    MODEL_PATH,
    compress=3,
)

print(f"Saved baseline artifact to: {MODEL_PATH}")


Saved baseline artifact to: C:\Users\nairs\OneDrive\Documents\Surabhi\Projects\Catch_Fraud\artifacts\models\sgd_baseline_v1.joblib
